In [17]:
import sys
# import logging
from pathlib import Path

# ── Path setup ────────────────────────────────────────────────────────────────
ROOT    = Path().resolve().parents[0]
SRC_DIR = ROOT / "src"
for p in [str(ROOT), str(SRC_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)
ROOT

WindowsPath('D:/Projects/Madstat/madstrat_backtest')

In [ ]:
# ── Path setup ────────────────────────────────────────────────────────────────
# ROOT    = Path().resolve().parents[1]
# SRC_DIR = ROOT / "src"
# for p in [str(ROOT), str(SRC_DIR)]:
#     if p not in sys.path:
#         sys.path.insert(0, p)

from src.data.process_data import DataProcessor, RAW_DATA_DIR, PROCESSED_DIR

In [69]:
processor = DataProcessor()

In [ ]:
import pandas as pd
raw_data_path = ROOT / "data" / "raw"
# list all CSV files in the raw data directory
daily_file = [f.name for f in raw_data_path.glob("*.csv") if "1d" in f.name]
intraday_file = [f.name for f in raw_data_path.glob("*.csv") if "15m" in f.name]

# file_names
# daily = pd.read_csv(ROOT / "data" / "raw" / daily_file[0])
# df = pd.read_csv(ROOT / "data" / "raw" / intraday_file[0])


In [84]:
daily = processor._load(ROOT / "data" / "raw" / daily_file[0])
df = processor._load(ROOT / "data" / "raw" / intraday_file[0])

2026-05-05 00:27:30,099 | INFO | Loaded 42 rows | 2026-03-04 03:00:00 -> 2026-05-01 02:00:00
2026-05-05 00:27:30,133 | INFO | Loaded 3876 rows | 2026-03-04 00:00:00 -> 2026-05-02 01:45:00


In [53]:
# Lowercase all column names immediately
daily.columns = [c.lower() for c in daily.columns]

# ── Locate and set the datetime index ─────────────────────────────────
# Priority order of candidate column names
dt_candidates = ["datetime", "date", "time", "timestamp"]

dt_col = None
for candidate in dt_candidates:
    if candidate in daily.columns:
        dt_col = candidate
        break

if dt_col:
    # Datetime is a regular column — parse and set as index
    daily[dt_col] = pd.to_datetime(daily[dt_col])
    daily = daily.set_index(dt_col)
    daily.index.name = "datetime"
else:
    # Assume first column is the datetime index (yfinance layout)
    df = pd.read_csv(raw_data_path, index_col=0, parse_dates=True)
    df.columns = [c.lower() for c in df.columns]

In [54]:
daily

,open,high,low,close,volume
datetime,,,,,
2026-03-04 03:00:00,5106.335,5206.265,5105.370,5140.945,1224059.0
2026-03-05 03:00:00,5152.485,5195.220,5050.905,5081.380,1240770.0
2026-03-06 03:00:00,5090.810,5176.630,5063.215,5171.500,1161417.0
2026-03-09 02:00:00,5171.950,5171.950,5015.045,5137.910,1394107.0
2026-03-10 02:00:00,5128.805,5238.775,5117.355,5192.510,833109.0
2026-03-11 02:00:00,5191.790,5223.385,5149.155,5176.480,630037.0
2026-03-12 02:00:00,5178.045,5191.790,5055.125,5079.255,804529.0
2026-03-13 02:00:00,5083.865,5128.475,5009.745,5020.600,695124.0
2026-03-16 02:00:00,5014.955,5038.145,4967.775,5006.565,865975.0


In [56]:
# Previous week levels built from daily candles
weekly = (
    daily["high"].resample("W-MON", label="left", closed="left").max()
    .to_frame("weekly_high")
    .join(
        daily["low"].resample("W-MON", label="left", closed="left")
        .min()
        .rename("weekly_low")
    )
    .dropna()
)


weekly["PWH"]   = weekly["weekly_high"].shift(1)
weekly["PWL"]   = weekly["weekly_low"].shift(1)

In [57]:
weekly

,weekly_high,weekly_low,PWH,PWL
datetime,,,,
2026-03-02,5206.265,5050.905,NaN,NaN
2026-03-09,5238.775,5009.745,5206.265,5050.905
2026-03-16,5044.900,4477.540,5238.775,5009.745
2026-03-23,4602.645,4099.125,5044.900,4477.540
2026-03-30,4800.800,4417.430,4602.645,4099.125
2026-04-06,4858.215,4600.850,4800.800,4417.430
2026-04-13,4891.540,4644.465,4858.215,4600.850
2026-04-20,4833.405,4658.030,4891.540,4644.465
2026-04-27,4729.965,4510.095,4833.405,4658.030


In [58]:
weekly_aligned = weekly[["PWH", "PWL"]].reindex(df.index, method="ffill")

In [60]:
daily['pwh'] = weekly['PWH']
daily

,open,high,low,close,volume,pwh
datetime,,,,,,
2026-03-04 03:00:00,5106.335,5206.265,5105.370,5140.945,1224059.0,NaN
2026-03-05 03:00:00,5152.485,5195.220,5050.905,5081.380,1240770.0,NaN
2026-03-06 03:00:00,5090.810,5176.630,5063.215,5171.500,1161417.0,NaN
2026-03-09 02:00:00,5171.950,5171.950,5015.045,5137.910,1394107.0,NaN
2026-03-10 02:00:00,5128.805,5238.775,5117.355,5192.510,833109.0,NaN
2026-03-11 02:00:00,5191.790,5223.385,5149.155,5176.480,630037.0,NaN
2026-03-12 02:00:00,5178.045,5191.790,5055.125,5079.255,804529.0,NaN
2026-03-13 02:00:00,5083.865,5128.475,5009.745,5020.600,695124.0,NaN
2026-03-16 02:00:00,5014.955,5038.145,4967.775,5006.565,865975.0,NaN


In [49]:
daily

,open,high,low,close,volume,PWH,PWL
datetime,,,,,,,
2026-03-04 03:00:00,5106.335,5206.265,5105.370,5140.945,1224059.0,NaN,NaN
2026-03-05 03:00:00,5152.485,5195.220,5050.905,5081.380,1240770.0,NaN,NaN
2026-03-06 03:00:00,5090.810,5176.630,5063.215,5171.500,1161417.0,NaN,NaN
2026-03-09 02:00:00,5171.950,5171.950,5015.045,5137.910,1394107.0,NaN,NaN
2026-03-10 02:00:00,5128.805,5238.775,5117.355,5192.510,833109.0,NaN,NaN
2026-03-11 02:00:00,5191.790,5223.385,5149.155,5176.480,630037.0,NaN,NaN
2026-03-12 02:00:00,5178.045,5191.790,5055.125,5079.255,804529.0,NaN,NaN
2026-03-13 02:00:00,5083.865,5128.475,5009.745,5020.600,695124.0,NaN,NaN
2026-03-16 02:00:00,5014.955,5038.145,4967.775,5006.565,865975.0,NaN,NaN


In [62]:
weekly = pd.DataFrame({
    "open":  df["open"].resample("W").first(),
    "high":  df["high"].resample("W").max(),
    "low":   df["low"].resample("W").min(),
    "close": df["close"].resample("W").last(),
}).dropna()


In [63]:
weekly

,open,high,low,close
datetime,,,,
2026-03-01,5173.570,5281.155,5130.235,5278.510
2026-03-08,5368.530,5419.660,4996.275,5171.500
2026-03-15,5171.950,5238.775,5009.745,5020.600
2026-03-22,5014.955,5044.900,4477.540,4497.480
2026-03-29,4472.240,4602.645,4099.125,4493.685
2026-04-05,4512.540,4800.800,4417.430,4676.745
2026-04-12,4638.250,4858.215,4600.850,4749.685
2026-04-19,4670.425,4891.540,4644.465,4831.610
2026-04-26,4774.815,4833.405,4658.030,4709.750


In [86]:
df = processor._add_weekly_levels(df, daily)
df

2026-05-05 00:27:44,470 | INFO | Added: PWH, PWL, PW_EQ
2026-05-05 00:27:44,496 | INFO | Added: WH, WL


,open,high,low,close,volume,PWH,PWL,PW_EQ,WH,WL
datetime,,,,,,,,,,
2026-03-04 00:00:00,5136.130,5149.055,5109.915,5112.470,20183.0,NaN,NaN,NaN,5149.055,5109.915
2026-03-04 00:15:00,5112.380,5121.360,5102.030,5106.825,16329.0,NaN,NaN,NaN,5149.055,5102.030
2026-03-04 00:30:00,5106.830,5119.230,5099.430,5119.120,23260.0,NaN,NaN,NaN,5149.055,5099.430
2026-03-04 00:45:00,5119.135,5128.550,5107.190,5113.395,22060.0,NaN,NaN,NaN,5149.055,5099.430
2026-03-04 01:00:00,5113.385,5127.075,5112.240,5119.125,20531.0,NaN,NaN,NaN,5149.055,5099.430
...,...,...,...,...,...,...,...,...,...,...
2026-05-02 00:45:00,4611.880,4615.450,4609.015,4610.920,14902.0,4833.405,4658.03,4745.7175,4729.965,4510.095
2026-05-02 01:00:00,4610.905,4612.820,4607.220,4611.500,4831.0,4833.405,4658.03,4745.7175,4729.965,4510.095
2026-05-02 01:15:00,4611.555,4616.225,4609.955,4613.430,3419.0,4833.405,4658.03,4745.7175,4729.965,4510.095


In [88]:
# Forward-fill the intraday PWH/PWL onto the daily index
# This avoids the resample mismatch when daily index != intraday buckets
daily["PWH"] = df["PWH"].reindex(daily.index, method="ffill")
daily["PWL"] = df["PWL"].reindex(daily.index, method="ffill")
# If exact daily timestamps don't exist in intraday, use nearest prior bar
if daily["PWH"].isna().all():
    daily["PWH"] = df["PWH"].reindex(
        daily.index, method="ffill", tolerance=pd.Timedelta("1D")
    )
    daily["PWL"] = df["PWL"].reindex(
        daily.index, method="ffill", tolerance=pd.Timedelta("1D")
    )
d = daily.dropna(subset=["PWH", "PWL"])

d["FBR_bull"] = (d["high"]  > d["PWH"]) & (d["close"] <= d["PWH"])
d["FBR_bear"] = (d["low"]   < d["PWL"]) & (d["close"] >= d["PWL"])
d["FBR"]      = d["FBR_bull"] | d["FBR_bear"]
d


,open,high,low,close,volume,PWH,PWL,FBR_bull,FBR_bear,FBR
datetime,,,,,,,,,,
2026-03-10 02:00:00,5128.805,5238.775,5117.355,5192.510,833109.0,5206.265,5050.905,True,False,True
2026-03-11 02:00:00,5191.790,5223.385,5149.155,5176.480,630037.0,5206.265,5050.905,True,False,True
2026-03-12 02:00:00,5178.045,5191.790,5055.125,5079.255,804529.0,5206.265,5050.905,False,False,False
2026-03-13 02:00:00,5083.865,5128.475,5009.745,5020.600,695124.0,5206.265,5050.905,False,False,False
2026-03-16 02:00:00,5014.955,5038.145,4967.775,5006.565,865975.0,5206.265,5050.905,False,False,False
2026-03-17 02:00:00,5011.110,5044.900,4973.720,5005.725,665101.0,5238.775,5009.745,False,False,False
2026-03-18 02:00:00,5001.960,5016.530,4806.845,4818.415,908731.0,5238.775,5009.745,False,False,False
2026-03-19 02:00:00,4824.645,4867.225,4502.365,4650.720,1777919.0,5238.775,5009.745,False,False,False
2026-03-20 02:00:00,4659.990,4736.175,4477.540,4497.480,1538222.0,5238.775,5009.745,False,False,False
